## CELL 1 dataloader

In [ ]:
# import os
# import sys

# # 1. Search the entire Kaggle input directory for your specific file
# target_file = 'aies_dataloader.py'
# found_path = None

# for root, dirs, files in os.walk('/kaggle/input'):
#     if target_file in files:
#         found_path = root
#         break

# if found_path:
#     print(f"🎯 Found dataset folder at: {found_path}")
    
#     # 2. Add that exact folder to Python's path
#     if found_path not in sys.path:
#         sys.path.append(found_path)
    
#     # 3. Now import your classes
#     from aies_dataloader import CGCD_Master_Dataset, DATA_PATHS
#     print("✅ Data Engine Successfully Imported!")
# else:
#     print(f"❌ Error: Could not find {target_file}. Please check the right sidebar to ensure the dataset is attached.")

In [ ]:
import os
import sys
from unittest.mock import MagicMock

# 1. THE SMART MOCK: Hardcoded with your verified Kaggle paths
def fake_dataset_download(handle):
    h = handle.lower()
    
    # Path verified in previous step
    if "cub" in h: 
        return "/kaggle/input/datasets/wenewone/cub2002011/CUB_200_2011"
    
    # Path updated based on your latest error message
    if "tiny" in h: 
        # return "/kaggle/input/datasets/akash2sharma/tiny-imagenet/tiny-imagenet-200"
        return "/kaggle/input/datasets/akash2sharma/tiny-imagenet/tiny-imagenet-200/tiny-imagenet-200"
    # Adjusted based on the pattern of your other datasets
    if "imagenet100" in h: 
        return "/kaggle/input/datasets/ambityga/imagenet100"
        
    return "/kaggle/working/data"

mock_hub = MagicMock()
mock_hub.dataset_download = fake_dataset_download
sys.modules['kagglehub'] = mock_hub

# 2. SEARCH: Locate your script but skip the data folders to keep it fast
found_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    # Prune massive directories from search tree
    for ignore in ['tiny-imagenet', 'cub2002011', 'imagenet100', 'images']:
        if ignore in dirs: 
            dirs.remove(ignore) 

    if 'aies_dataloader.py' in files:
        found_path = root
        break

if found_path:
    if found_path not in sys.path: 
        sys.path.append(found_path)
    print(f"🎯 Found script at: {found_path}")
else:
    # Error handling for your specific career-critical script
    raise FileNotFoundError("❌ Script not found. Check the right sidebar for 'sparsity-dataloader-aies'.")

# 3. THE IMPORT: This triggers the audit loop in your script
from aies_dataloader import CGCD_Master_Dataset, DATA_PATHS

print("✅ Script imported successfully!")
print("Verified TINY Root:", DATA_PATHS["TINY"])

## CELL 2 dataset wrapper for HappyCgcd

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image

# ==============================================================================
# CELL 2: THE DATALOADER WRAPPER (Bridging Your Engine to Happy-CGCD)
# ==============================================================================
class ContrastiveLearningViewGenerator(object):
    def __init__(self, base_transform, n_views=2):
        self.base_transform = base_transform
        self.n_views = n_views

    def __call__(self, x):
        return [self.base_transform(x) for i in range(self.n_views)]

class HappyDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        # 1. Get the real index from your ablation engine
        real_idx = self.base_dataset.final_indices[idx]
        
        # 2. Safely grab the raw PIL image (bypassing your ToTensor)
        if self.base_dataset.dataset_name == "C100":
            image = self.base_dataset.image_paths_or_data[real_idx]
            image = Image.fromarray(image)
        else:
            img_path = self.base_dataset.image_paths_or_data[real_idx]
            image = Image.open(img_path).convert('RGB')
            
        label = self.base_dataset.all_targets[real_idx]
        
        # 3. Apply Happy's Contrastive Augmentations
        images = self.transform(image)
        
        # 4. Return exactly what Happy's online/offline loop expects
        return images, label, real_idx, 1 

# Standard transforms for 224x224 (ViT requirement)
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])
happy_train_transform = ContrastiveLearningViewGenerator(base_transform=base_transform, n_views=2)

print("✅ Happy-CGCD Dataset Wrapper Initialized!")

## CELL 3 simulaton of outputs table just to check

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# ==============================================================================
# CELL 3: THE TABLE 1 & TABLE 3 METRICS ENGINE (SIMULATION)
# ==============================================================================
def cluster_acc(y_true, y_pred, mask):
    """Hungarian matching algorithm from Happy's cluster_utils.py"""
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    mask = mask.astype(bool)
    
    # Linear assignment
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=int)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1

    ind = linear_sum_assignment(w.max() - w)
    ind = np.vstack(ind).T
    ind_map = {j: i for i, j in ind}
    
    total_acc = sum([w[i, j] for i, j in ind]) * 1.0 / y_pred.size
    
    # Split into Old and New
    old_classes_gt = set(y_true[mask])
    new_classes_gt = set(y_true[~mask])
    
    old_acc, total_old = 0, 0
    for i in old_classes_gt:
        old_acc += w[ind_map[i], i]
        total_old += sum(w[:, i])
    old_acc = (old_acc / total_old) * 100 if total_old > 0 else 0.0

    new_acc, total_new = 0, 0
    for i in new_classes_gt:
        if i in ind_map: # Handle cases where new classes haven't been discovered yet
            new_acc += w[ind_map[i], i]
        total_new += sum(w[:, i])
    new_acc = (new_acc / total_new) * 100 if total_new > 0 else 0.0

    return total_acc * 100, old_acc, new_acc

# ==============================================================================
# MOCK SIMULATION (Proving the logging structure works)
# ==============================================================================
print("="*100)
print(f"{'TABLE 1: STAGE-WISE CONTINUAL GCD ACCURACY (CIFAR-100)':^100}")
print("="*100)
print(f"{'Method':<15} | {'Stage 0':<12} | {'Stage 1 (All/Old/New)':<25} | {'...'} | {'Stage 5 (All/Old/New)'}")
print("-" * 100)

stage_metrics = {}

# Simulate running all stages
for stage in range(6):
    # 1. Initialize your custom data engine for the specific stage
    ds = CGCD_Master_Dataset("C100", DATA_PATHS["C100"], stage=stage, sparsity_level=0.90)
    
    # 2. Wrap it for Happy
    happy_loader = torch.utils.data.DataLoader(
        HappyDatasetWrapper(ds, happy_train_transform), 
        batch_size=128, shuffle=True
    )
    
    # [THIS IS WHERE `train_online(model, happy_loader)` WILL GO SHORTLY]
    
    # 3. Simulate outputs for evaluation
    y_true = np.array([ds.all_targets[idx] for idx in ds.final_indices])
    
    # Faking predictions to show how the table format works
    # (Pretending accuracy drops slightly each stage on Old, rises on New)
    base_acc = 0.90 - (stage * 0.05)
    y_pred = np.where(np.random.rand(len(y_true)) < base_acc, y_true, np.random.randint(0, 100, len(y_true)))
    
    # Define mask: Old classes vs New classes
    old_class_list = ds.base_classes
    mask = np.isin(y_true, old_class_list)
    
    all_a, old_a, new_a = cluster_acc(y_true, y_pred, mask)
    stage_metrics[stage] = {'all': all_a, 'old': old_a, 'new': new_a}

# --- Print Table 1 Format ---
t1_str = f"{'Happy (Ours)':<15} | {stage_metrics[0]['all']:.2f} (All) | "
for s in range(1, 6):
    t1_str += f"S{s}: {stage_metrics[s]['all']:.1f}/{stage_metrics[s]['old']:.1f}/{stage_metrics[s]['new']:.1f} | "
print(t1_str)

# --- Print Table 3 Format ---
print("\n" + "="*50)
print(f"{'TABLE 3: FORGETTING & DISCOVERY':^50}")
print("="*50)
# M_f = Old Acc at Stage 0 - Old Acc at Stage 5
M_f = stage_metrics[0]['old'] - stage_metrics[5]['old']
# M_d = New Acc at Stage 5
M_d = stage_metrics[5]['new']

print(f"{'Method':<15} | {'M_f ↓':<15} | {'M_d ↑':<15}")
print("-" * 50)
print(f"{'Happy (Ours)':<15} | {M_f:.2f}          | {M_d:.2f}")
print("="*50)

## CELL 4 - Happy-CGCD Neural Network & Math Classes

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.cluster import KMeans

# ==============================================================================
# CELL 4: HAPPY-CGCD MODEL & LOSS ARCHITECTURE
# ==============================================================================
class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim, use_bn=False, norm_last_layer=True, nlayers=3, hidden_dim=2048, bottleneck_dim=256):
        super().__init__()
        nlayers = max(nlayers, 1)
        if nlayers == 1:
            self.mlp = nn.Linear(in_dim, bottleneck_dim)
        else:
            layers = [nn.Linear(in_dim, hidden_dim)]
            if use_bn: layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.GELU())
            for _ in range(nlayers - 2):
                layers.append(nn.Linear(hidden_dim, hidden_dim))
                if use_bn: layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.GELU())
            layers.append(nn.Linear(hidden_dim, bottleneck_dim))
            self.mlp = nn.Sequential(*layers)
        self.apply(self._init_weights)
        self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)
        if norm_last_layer:
            self.last_layer.weight_g.requires_grad = False

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x_proj = self.mlp(x)
        x_norm = nn.functional.normalize(x_proj, dim=-1, p=2)
        logits = self.last_layer(x_norm)
        return x_proj, logits

class SupConLoss(torch.nn.Module):
    def __init__(self, temperature=0.07, base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        device = features.device
        if len(features.shape) > 3: features = features.view(features.shape[0], features.shape[1], -1)
        batch_size = features.shape[0]
        if labels is not None:
            labels = labels.contiguous().view(-1, 1)
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)
            
        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        anchor_feature = contrast_feature
        anchor_count = contrast_count

        anchor_dot_contrast = torch.div(torch.matmul(anchor_feature, contrast_feature.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()
        mask = mask.repeat(anchor_count, contrast_count)
        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size * anchor_count).view(-1, 1).to(device), 0)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        return loss.view(anchor_count, batch_size).mean()

def info_nce_logits(features, n_views=2, temperature=1.0, device='cuda'):
    b_ = 0.5 * int(features.size(0))
    labels = torch.cat([torch.arange(b_) for i in range(n_views)], dim=0)
    labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float().to(device)
    features = F.normalize(features, dim=1)
    similarity_matrix = torch.matmul(features, features.T)
    mask = torch.eye(labels.shape[0], dtype=torch.bool).to(device)
    labels = labels[~mask].view(labels.shape[0], -1)
    similarity_matrix = similarity_matrix[~mask].view(similarity_matrix.shape[0], -1)
    positives = similarity_matrix[labels.bool()].view(labels.shape[0], -1)
    negatives = similarity_matrix[~labels.bool()].view(similarity_matrix.shape[0], -1)
    logits = torch.cat([positives, negatives], dim=1)
    labels = torch.zeros(logits.shape[0], dtype=torch.long).to(device)
    logits = logits / temperature
    return logits, labels

class DistillLoss(nn.Module):
    def __init__(self, warmup_teacher_temp_epochs, nepochs, ncrops=2, warmup_teacher_temp=0.07, teacher_temp=0.04, student_temp=0.1):
        super().__init__()
        self.student_temp = student_temp
        self.ncrops = ncrops
        self.teacher_temp_schedule = np.concatenate((
            np.linspace(warmup_teacher_temp, teacher_temp, warmup_teacher_temp_epochs),
            np.ones(nepochs - warmup_teacher_temp_epochs) * teacher_temp
        ))

    def forward(self, student_output, teacher_output, epoch):
        student_out = student_output / self.student_temp
        student_out = student_out.chunk(self.ncrops)
        temp = self.teacher_temp_schedule[epoch]
        teacher_out = F.softmax(teacher_output / temp, dim=-1)
        teacher_out = teacher_out.detach().chunk(2)
        total_loss, n_loss_terms = 0, 0
        for iq, q in enumerate(teacher_out):
            for v in range(len(student_out)):
                if v == iq: continue
                loss = torch.sum(-q * F.log_softmax(student_out[v], dim=-1), dim=-1)
                total_loss += loss.mean()
                n_loss_terms += 1
        return total_loss / n_loss_terms

class ProtoAugManager:
    def __init__(self, feature_dim, batch_size, hardness_temp, radius_scale, device):
        self.feature_dim = feature_dim
        self.batch_size = batch_size
        self.device = device
        self.hardness_temp = hardness_temp
        self.radius_scale = radius_scale
        self.prototypes = None
        self.mean_similarity = None
        self.radius = 0

    def compute_proto_aug_loss(self, model):
        prototypes = F.normalize(self.prototypes, dim=-1, p=2).to(self.device)
        sampling_prob = F.softmax(self.mean_similarity / self.hardness_temp, dim=-1).cpu().numpy()
        prototypes_labels = np.random.choice(len(prototypes), size=(self.batch_size,), replace=True, p=sampling_prob)
        prototypes_labels = torch.from_numpy(prototypes_labels).long().to(self.device)
        prototypes_sampled = prototypes[prototypes_labels]
        prototypes_augmented = prototypes_sampled + torch.randn((self.batch_size, self.feature_dim), device=self.device) * self.radius * self.radius_scale
        _, prototypes_output = model[1](prototypes_augmented)
        return nn.CrossEntropyLoss()(prototypes_output / 0.1, prototypes_labels)

def get_params_groups(model):
    regularized, not_regularized = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if name.endswith(".bias") or len(param.shape) == 1: not_regularized.append(param)
        else: regularized.append(param)
    return [{'params': regularized}, {'params': not_regularized, 'weight_decay': 0.}]

print("✅ Happy-CGCD Architecture Classes Loaded!")

## CELL 5 - The Main Master Training Loop (Stage 0 to 5)

## updated cell5 for short medium hyper params

In [ ]:
from torch.optim import SGD
from copy import deepcopy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# ==============================================================================
# CELL 5: THE HAPPY-CGCD TRAINING PIPELINE (Dynamic Multi-Dataset Version)
# ==============================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---> 🎛️ THE MASTER EXECUTION TOGGLE 🎛️ <---
RUN_MODE = "2_HOUR_TEST"  # Options: "FAST_TEST", "1_HOUR_TEST", "2_HOUR_TEST", "MEDIUM_TEST", "FULL_RUN"
TARGET_DATASET = "C100"   # Change this to "CUB", "TINY", or "IN100"

print("\n" + "="*80)
print(f"🚀 INITIALIZING HAPPY-CGCD ENGINE")
print(f"Dataset: {TARGET_DATASET} | Mode: {RUN_MODE}")
print("="*80)

# 1. BASH COMMAND HYPERPARAMETER ROUTER
DATASET_CONFIGS = {
    "C100":  {"base": 50,  "chunk": 10, "epochs_off": 100, "epochs_on": 30},
    "IN100": {"base": 50,  "chunk": 10, "epochs_off": 100, "epochs_on": 30},
    "TINY":  {"base": 100, "chunk": 20, "epochs_off": 100, "epochs_on": 30},
    "CUB":   {"base": 100, "chunk": 20, "epochs_off": 100, "epochs_on": 20} 
}

NUM_BASE_CLASSES = DATASET_CONFIGS[TARGET_DATASET]["base"]
CHUNK_SIZE = DATASET_CONFIGS[TARGET_DATASET]["chunk"]

BASH_EPOCHS_OFFLINE = DATASET_CONFIGS[TARGET_DATASET]["epochs_off"]
BASH_EPOCHS_ONLINE = DATASET_CONFIGS[TARGET_DATASET]["epochs_on"]

# 2. RUN_MODE MULTIPLIERS 
if RUN_MODE == "FAST_TEST":
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 1, 1, 3         
elif RUN_MODE == "1_HOUR_TEST":
    # 5 Offline + (1 Online * 5 Stages) = 10 Epochs Total 
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 5, 1, 999999
elif RUN_MODE == "2_HOUR_TEST":
    # 10 Offline + (2 Online * 5 Stages) = 20 Epochs Total (~2 hours runtime)
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 10, 5, 999999 
elif RUN_MODE == "MEDIUM_TEST":
    # 30 Offline + (10 Online * 5 Stages) = 80 Epochs Total (~8 hours runtime)
    EPOCHS_OFFLINE, EPOCHS_ONLINE, MAX_BATCHES = 30, 10, 999999  
else: # FULL_RUN
    # Matches official bash commands exactly (~12-13 hours)
    EPOCHS_OFFLINE = BASH_EPOCHS_OFFLINE
    EPOCHS_ONLINE = BASH_EPOCHS_ONLINE
    MAX_BATCHES = 999999 

# --- GLOBAL HYPERPARAMETERS ---
BATCH_SIZE = 128
LR_OFFLINE = 0.1     
LR_ONLINE = 0.01     
WEIGHT_DECAY = 5e-5
FEAT_DIM = 768

print(f"⚙️ Config Locked: Offline Epochs: {EPOCHS_OFFLINE} | Online Epochs: {EPOCHS_ONLINE}")

print(f"⚙️ Config Locked: Offline Epochs: {EPOCHS_OFFLINE} | Online Epochs: {EPOCHS_ONLINE}")

# 3. Load Pretrained DINO ViT
print(f"Initializing DINO ViT Backbone...")
backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')
for m in backbone.parameters(): m.requires_grad = False
for name, m in backbone.named_parameters():
    if 'blocks.11' in name: m.requires_grad = True

# 4. Master Tracking Variables
model_cur, model_pre = None, None
proto_aug_manager = ProtoAugManager(FEAT_DIM, BATCH_SIZE*2, 0.1, 1.0, DEVICE)
stage_metrics = {}

# ==============================================================================
# MAIN STAGE LOOP
# ==============================================================================
for stage in range(6):
    print(f"\n{'='*40}\n---> INITIALIZING STAGE {stage} <---\n{'='*40}")
    
    ds = CGCD_Master_Dataset(TARGET_DATASET, DATA_PATHS[TARGET_DATASET], stage=stage, sparsity_level=0.90)
    train_loader = torch.utils.data.DataLoader(HappyDatasetWrapper(ds, happy_train_transform), batch_size=BATCH_SIZE, drop_last=True, shuffle=True)
    
    if stage == 0:
        num_seen_classes = NUM_BASE_CLASSES
        num_cur_classes = NUM_BASE_CLASSES
        
        # --- OFFLINE STAGE 0 ---
        projector = DINOHead(in_dim=FEAT_DIM, out_dim=NUM_BASE_CLASSES)
        model_cur = nn.Sequential(backbone, projector).to(DEVICE)
        
        optimizer = SGD(get_params_groups(model_cur), lr=LR_OFFLINE, momentum=0.9, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_OFFLINE)
        cluster_criterion = DistillLoss(min(30, EPOCHS_OFFLINE), EPOCHS_OFFLINE)
        
        model_cur.train()
        for epoch in range(EPOCHS_OFFLINE):
            for batch_idx, (images, labels, uq_idxs, _) in enumerate(train_loader):
                if batch_idx >= MAX_BATCHES: break 
                
                images, labels = torch.cat(images, dim=0).to(DEVICE), labels.to(DEVICE)
                proj, out = model_cur(images)
                teacher_out = out.detach()
                
                sup_logits = torch.cat([f for f in (out / 0.1).chunk(2)], dim=0)
                sup_labels = torch.cat([labels for _ in range(2)], dim=0)
                cls_loss = nn.CrossEntropyLoss()(sup_logits, sup_labels)
                cluster_loss = cluster_criterion(out, teacher_out, epoch)
                con_logits, con_labels = info_nce_logits(features=proj)
                con_loss = nn.CrossEntropyLoss()(con_logits, con_labels)
                
                loss = (0.65 * cluster_loss) + (0.35 * cls_loss) + (0.65 * con_loss)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                if batch_idx % 20 == 0: print(f"Stage 0 | Epoch {epoch} | Batch {batch_idx} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.4f}")
            
            # CRITICAL: Step the scheduler at the end of the epoch!
            scheduler.step()
                
        proto_aug_manager.prototypes = torch.randn((NUM_BASE_CLASSES, FEAT_DIM))
        proto_aug_manager.mean_similarity = torch.ones(NUM_BASE_CLASSES)

    else:
        num_seen_classes = NUM_BASE_CLASSES + ((stage - 1) * CHUNK_SIZE)
        num_cur_classes = num_seen_classes + CHUNK_SIZE
        
        # --- ONLINE STAGE 1 TO 5 ---
        model_pre = deepcopy(model_cur)
        model_pre.eval()
        
        projector_cur = DINOHead(in_dim=FEAT_DIM, out_dim=num_cur_classes)
        with torch.no_grad():
            projector_cur.last_layer.weight_v.data[:num_seen_classes] = model_pre[1].last_layer.weight_v.data[:num_seen_classes]
            projector_cur.last_layer.weight.data[:num_seen_classes] = model_pre[1].last_layer.weight.data[:num_seen_classes]
        
        model_cur = nn.Sequential(backbone, projector_cur).to(DEVICE)
        
        optimizer = SGD(get_params_groups(model_cur), lr=LR_ONLINE, momentum=0.9, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_ONLINE)
        cluster_criterion = DistillLoss(min(10, EPOCHS_ONLINE), EPOCHS_ONLINE)
        
        model_cur.train()
        for epoch in range(EPOCHS_ONLINE):
            for batch_idx, (images, labels, uq_idxs, _) in enumerate(train_loader):
                if batch_idx >= MAX_BATCHES: break 
                
                images = torch.cat(images, dim=0).to(DEVICE)
                proj, out = model_cur(images)
                teacher_out = out.detach()
                
                cluster_loss = cluster_criterion(out, teacher_out, epoch)
                con_logits, con_labels = info_nce_logits(features=proj)
                con_loss = nn.CrossEntropyLoss()(con_logits, con_labels)
                
                proto_loss = proto_aug_manager.compute_proto_aug_loss(model_cur)
                with torch.no_grad(): feats_pre = model_pre[0](images)
                feats = model_cur[0](images)
                feat_distill_loss = (F.normalize(feats, dim=-1) - F.normalize(feats_pre, dim=-1)).pow(2).sum() / len(feats)
                
                loss = cluster_loss + con_loss + proto_loss + feat_distill_loss
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                if batch_idx % 20 == 0: print(f"Stage {stage} | Epoch {epoch} | Batch {batch_idx} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.4f}")
            
            # CRITICAL: Step the scheduler at the end of the epoch!
            scheduler.step()
                
        proto_aug_manager.prototypes = torch.randn((num_cur_classes, FEAT_DIM))
        proto_aug_manager.mean_similarity = torch.ones(num_cur_classes)

    # --- EVALUATION BLOCK ---
    print(f"Evaluating Stage {stage}...")
    model_cur.eval()
    eval_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])
    
    y_true, y_pred = [], []
    eval_indices = ds.final_indices[:200] if RUN_MODE == "FAST_TEST" else ds.final_indices
    
    with torch.no_grad():
        for idx in eval_indices:
            if ds.dataset_name == "C100": img = Image.fromarray(ds.image_paths_or_data[idx])
            else: img = Image.open(ds.image_paths_or_data[idx]).convert('RGB')
            
            img_t = eval_transform(img).unsqueeze(0).to(DEVICE)
            _, logits = model_cur(img_t)
            
            y_true.append(ds.all_targets[idx])
            y_pred.append(logits.argmax(1).item())
            
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = np.isin(y_true, ds.base_classes)
    
    all_a, old_a, new_a = cluster_acc(y_true, y_pred, mask)
    stage_metrics[stage] = {'all': all_a, 'old': old_a, 'new': new_a}

# --- FINAL TABLE PRINTS ---
print("\n" + "="*100)
print(f"{'TABLE 1: STAGE-WISE CONTINUAL GCD ACCURACY (' + TARGET_DATASET + ')':^100}")
print("="*100)
t1_str = f"{'Happy (Ours)':<15} | {stage_metrics[0]['all']:.2f} (All) | "
for s in range(1, 6): t1_str += f"S{s}: {stage_metrics[s]['all']:.1f}/{stage_metrics[s]['old']:.1f}/{stage_metrics[s]['new']:.1f} | "
print(t1_str)

print("\n" + "="*50)
print(f"{'TABLE 3: FORGETTING & DISCOVERY (' + TARGET_DATASET + ')':^50}")
print("="*50)
M_f = stage_metrics[0]['old'] - stage_metrics[5]['old']
M_d = stage_metrics[5]['new']
print(f"{'Method':<15} | {'M_f ↓':<15} | {'M_d ↑':<15}")
print(f"{'Happy (Ours)':<15} | {M_f:.2f}          | {M_d:.2f}")
print("="*50)